# Train the transformer role classifier on Colab

Choose **Runtime → Change runtime type → GPU** (a T4 is sufficient for the configured microbatches) and run the cells in order. This notebook supports Colab's Python 3.13 runtime and keeps its preinstalled CUDA PyTorch.

The two candidates fully fine-tune **InLegalBERT** and **DeBERTa-v3-base**. These are non-generative pretrained language encoders used for seven-label classification. Each target sentence receives same-judgment previous/next context; only target-token representations are mean-pooled before dropout and a new linear head. All target tokens are retained in chunks of at most 384 tokens, within a 512-token input window. Chunk losses and predictions are weighted by target-token share. No API key or native sent2vec installation is needed.

The fixed seed-42 split has **32 fitting judgments, eight validation judgments and ten reserved test judgments**. Validation macro F1 across all seven roles selects the candidate and epoch; accuracy breaks ties. A fresh model then trains on all 40 training judgments for the selected epoch count, retaining the original six-epoch schedule horizon. The separate evaluation cell scores the ten reserved judgments after the model is frozen. Keep model selection fixed after viewing test results.

The older Hier-BiLSTM-CRF baseline reached 65.06% validation accuracy and recognized zero of 99 lower-court sentences. This notebook records measured transformer results; it assumes no performance improvement.

## Install the two hosted-runtime dependencies

Keep Colab's PyTorch, NumPy and scikit-learn. Do not install the repository's local Python 3.11 requirements in this hosted runtime.

In [ ]:
import subprocess
import sys

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--quiet',
    'transformers==4.57.1', 'sentencepiece==0.2.1',
], check=True)

## Verify the GPU and fetch the pinned source

The repository checkout and pretrained encoders are pinned independently. A fresh temporary workspace keeps outputs from different runs separate. Rerunning a stage cell watches the existing job instead of launching a duplicate.

In [ ]:
from datetime import datetime, timezone
from importlib.metadata import version
import hashlib
import json
from pathlib import Path
import platform
import tempfile
import time
from zipfile import ZipFile, ZIP_DEFLATED

import google.colab
import torch

assert torch.cuda.is_available(), 'Select a Colab GPU runtime before continuing.'
source_revision = '3c861b6d46438406f62ff0845993ed19bcf3fab8'
repository = 'https://github.com/SrikarKorrapati/Extractive-Summarisation-of-Legal-Document.git'
if 'transformer_work' not in globals():
    transformer_work = Path(tempfile.mkdtemp(prefix='transformer-roles-', dir='/content'))
    transformer_started = datetime.now(timezone.utc).isoformat()
    transformer_jobs = {}
transformer_source = transformer_work / 'source'
transformer_cache = transformer_work / 'encoder-cache'
if not transformer_source.exists():
    subprocess.run(['git', 'init', '--quiet', str(transformer_source)], check=True)
    subprocess.run([
        'git', 'fetch', '--quiet', '--depth=1', repository, source_revision,
    ], cwd=transformer_source, check=True)
    subprocess.run([
        'git', 'checkout', '--quiet', '--detach', 'FETCH_HEAD',
    ], cwd=transformer_source, check=True)
assert subprocess.check_output([
    'git', 'rev-parse', 'HEAD',
], cwd=transformer_source, text=True).strip() == source_revision
assert not subprocess.check_output([
    'git', 'status', '--porcelain', '--untracked-files=no',
], cwd=transformer_source)
print('Workspace:', transformer_work)
print('Python:', platform.python_version(), 'PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))
print('Transformers:', version('transformers'), 'SentencePiece:', version('sentencepiece'))


def launch_stage(name, module, arguments, output):
    if name in transformer_jobs:
        return transformer_jobs[name]
    if output.exists() and any(output.iterdir()):
        raise RuntimeError(f'Output already exists: {output}. Inspect it before restarting.')
    log = transformer_work / (name + '.log')
    if log.exists():
        raise RuntimeError(f'Job log already exists: {log}. Do not launch a duplicate job.')
    with log.open('x') as stream:
        process = subprocess.Popen([
            sys.executable, '-u', '-m', module, *map(str, arguments),
        ], cwd=transformer_source, stdout=stream, stderr=subprocess.STDOUT)
    transformer_jobs[name] = (process, log)
    return process, log


def watch(process, log):
    offset = 0
    while True:
        with log.open() as stream:
            stream.seek(offset)
            recent = stream.read()
            offset = stream.tell()
        if recent:
            print(recent, end='', flush=True)
        status = process.poll()
        if status is not None:
            if status:
                raise RuntimeError(f'Job exited with status {status}; inspect {log}')
            return
        time.sleep(10)

## Select using the 32 fitting and eight validation judgments

Run the two fixed configurations for at most six epochs each, with early-stopping patience two. Both use full encoder fine-tuning, FP16, gradient checkpointing, AdamW and fitting-label class weights. InLegalBERT uses microbatch 4 with eight accumulation steps; DeBERTa uses microbatch 2 with 16 accumulation steps. The validation reports include all seven roles and lower-court precision, recall, F1 and support.

No test metrics are computed during selection. The selected validation checkpoint stays in the tuning workspace and is used only to verify the selection before fresh refitting.

In [ ]:
transformer_tuning = transformer_work / 'tuning'
process, log = launch_stage(
    'tune', 'legal_lexrank.transformer_role_training',
    ['tune', '--data', transformer_source, '--output', transformer_tuning,
     '--cache-dir', transformer_cache, '--device', 'cuda',
     '--max-epochs', '6', '--patience', '2'],
    transformer_tuning,
)
watch(process, log)
selection = json.loads((transformer_tuning / 'selection.json').read_text())
chosen = selection['selected']
print('Selected encoder:', chosen['config']['encoder_model'])
print('Pretrained revision:', chosen['config']['encoder_revision'])
print('Selected epoch:', chosen['best_epoch'])
print('Validation macro F1:', chosen['validation']['macro avg']['f1-score'])
print('Validation accuracy:', chosen['validation']['accuracy'])
print('Lower-court validation:', chosen['validation']['Ruling by Lower Court'])

## Refit a fresh final model on all 40 training judgments

This stage reloads the selected original pretrained encoder and initializes a new head. It uses the selected epoch count and the same six-epoch schedule horizon, with class weights recalculated from the 40 training judgments. It does not continue from the validation checkpoint or evaluate any split.

In [ ]:
transformer_final = transformer_work / 'final'
process, log = launch_stage(
    'refit', 'legal_lexrank.transformer_role_training',
    ['refit', '--data', transformer_source, '--output', transformer_final,
     '--selection', transformer_tuning / 'selection.json',
     '--cache-dir', transformer_cache, '--device', 'cuda'],
    transformer_final,
)
watch(process, log)
final_protocol = json.loads((transformer_final / 'protocol.json').read_text())
assert final_protocol['format'] == 'legal-lexrank-role-study-v4'
assert len(final_protocol['case_ids']['train']) == 40
assert final_protocol['epochs'] == chosen['best_epoch']
assert final_protocol['schedule_epochs'] == 6
print('Frozen final classifier:', transformer_final / 'role_labeller.pt')
print('Final epochs:', final_protocol['epochs'], 'Schedule horizon:', final_protocol['schedule_epochs'])

## Evaluate the frozen model on the ten reserved judgments once

Run this cell only after refitting finishes. The evaluator verifies the final checkpoint and protocol, then writes one prediction per annotated sentence, the seven-role classification report and the confusion matrix. This step does not train the model. Test results must not change the chosen encoder, hyperparameters or epoch count.

In [ ]:
transformer_evaluation = transformer_final / 'test-evaluation'
process, log = launch_stage(
    'test-evaluation', 'legal_lexrank.evaluate_roles',
    ['--data', transformer_source, '--role-run', transformer_final,
     '--output', transformer_evaluation, '--cache', transformer_cache,
     '--device', 'cuda'],
    transformer_evaluation,
)
watch(process, log)
test_report = json.loads((transformer_evaluation / 'role_metrics.json').read_text())['test']
print('Held-out accuracy:', test_report['accuracy'])
print('Held-out macro F1:', test_report['macro avg']['f1-score'])
print('Held-out lower-court report:', test_report['Ruling by Lower Court'])

## Optional: run the full summary benchmark on Colab

Skip this cell if you only need the trained role classifier. Run it after the frozen role evaluation to compare **all four summary backends**, using the fixed default weights and an unweighted baseline at 30%, 40% and 50% sentence budgets on the ten test judgments.

The published cache contains 40 similarity matrices and provenance, with no role labels or credentials. The cell verifies the archive checksum, test-case IDs, encoder implementation hash and every cached file before extracting it. Reusing these matrices means **no OpenAI API key is needed**. The frozen role classifier runs on the GPU; there is no additional training or model selection. Completed JSON/CSV reports are added to the final download.


In [ ]:
from pathlib import PurePosixPath
import stat
from urllib.request import urlretrieve

summary_benchmark_complete = False
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--quiet',
    'rouge-score==0.1.2', 'nltk==3.8.1', 'rapidfuzz==3.14.6',
], check=True)

summary_cache_url = (
    'https://github.com/SrikarKorrapati/Extractive-Summarisation-of-Legal-Document/'
    'releases/download/lexrank-test-similarities-v1/lexrank_test_similarities.zip'
)
summary_cache_sha256 = 'cc6dc74a7ba9b874ad9209bea617020908c10d223808d9b3d297e1e2629ae103'
summary_cache_archive = transformer_work / 'lexrank_test_similarities.zip'
if not summary_cache_archive.exists():
    urlretrieve(summary_cache_url, summary_cache_archive)
assert summary_cache_archive.stat().st_size == 5_674_544, 'Unexpected similarity archive size.'
assert hashlib.sha256(summary_cache_archive.read_bytes()).hexdigest() == summary_cache_sha256

summary_cache = transformer_work / 'summary-cache'
summary_backends = ('tfidf', 'sbert', 'legalbert', 'openai')
with ZipFile(summary_cache_archive) as archive:
    infos = archive.infolist()
    names = [info.filename for info in infos]
    assert len(names) == len(set(names)), 'Duplicate archive paths.'
    for info in infos:
        relative = PurePosixPath(info.filename)
        assert (not relative.is_absolute() and '..' not in relative.parts
                and relative.as_posix() == info.filename and chr(92) not in info.filename)
        assert not info.is_dir() and not stat.S_ISLNK(info.external_attr >> 16)
    manifest_bytes = archive.read('cache_manifest.json')
    summary_cache_manifest = json.loads(manifest_bytes)
    assert summary_cache_manifest['case_ids'] == final_protocol['case_ids']['test']
    encoder_source = transformer_source / 'legal_lexrank' / 'encoders.py'
    assert summary_cache_manifest['encoder_source_sha256'] == hashlib.sha256(encoder_source.read_bytes()).hexdigest()
    entries = summary_cache_manifest['files']
    expected_pairs = {(backend, case_id) for backend in summary_backends
                      for case_id in final_protocol['case_ids']['test']}
    assert len(entries) == 40 and {(row['backend'], row['case_id']) for row in entries} == expected_pairs
    entry_paths = [row['path'] for row in entries]
    assert len(entry_paths) == len(set(entry_paths))
    assert set(names) == {'cache_manifest.json', *entry_paths}
    verified_files = {'cache_manifest.json': manifest_bytes}
    for row in entries:
        relative = PurePosixPath(row['path'])
        assert (len(relative.parts) == 4 and relative.parts[:2] == ('similarities', 'similarity')
                and relative.parts[2] == row['backend'] and relative.suffix == '.npy'
                and len(relative.stem) == 64 and all(c in '0123456789abcdef' for c in relative.stem))
        data = archive.read(row['path'])
        assert hashlib.sha256(data).hexdigest() == row['sha256'], f"Cache checksum mismatch: {row['path']}"
        verified_files[row['path']] = data
    # Extract only verified regular files into this run's cache directory.
    summary_cache.mkdir(exist_ok=True)
    cache_root = summary_cache.resolve()
    for name, data in verified_files.items():
        destination = summary_cache.joinpath(*PurePosixPath(name).parts)
        assert destination.resolve().is_relative_to(cache_root), 'Cache path escapes extraction directory.'
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(data)
print('Verified 40 similarity matrices for the ten reserved judgments.')

summary_benchmark = transformer_final / 'summary-benchmark'
process, log = launch_stage(
    'summary-benchmark', 'legal_lexrank',
    ['--data', transformer_source, '--role-run', transformer_final,
     '--output', summary_benchmark, '--cache', summary_cache, '--device', 'cuda'],
    summary_benchmark,
)
watch(process, log)
summary_benchmark_reports = (
    'protocol.json', 'tfidf.json', 'sbert.json', 'legalbert.json', 'openai.json',
    'role_metrics.json', 'metrics.csv',
)
assert all((summary_benchmark / name).is_file() for name in summary_benchmark_reports)
summary_benchmark_complete = True
print((summary_benchmark / 'metrics.csv').read_text())


## Download the final model and its audit bundle

The archive includes only the refitted model, protocol, selection, training/validation histories, test evaluation and runtime metadata, plus the optional summary benchmark reports when completed. Candidate models and the selected validation checkpoint are excluded. The bundle contains complete fine-tuned weights; the small pinned tokenizer files are fetched and cached when the classifier is first loaded locally.

In [ ]:
from google.colab import files

runtime = {
    'platform': 'Google Colab',
    'device': 'cuda',
    'gpu': torch.cuda.get_device_name(0),
    'gpu_memory_bytes': torch.cuda.get_device_properties(0).total_memory,
    'python': platform.python_version(),
    'torch': str(torch.__version__),
    'cuda': torch.version.cuda,
    'cudnn': torch.backends.cudnn.version(),
    'packages': {name: version(name) for name in (
        'transformers', 'sentencepiece', 'huggingface-hub', 'tokenizers',
        'safetensors', 'numpy', 'scipy', 'scikit-learn', 'filelock',
    )},
    'source_revision': source_revision,
    'encoder_model': final_protocol['config']['encoder_model'],
    'encoder_revision': final_protocol['config']['encoder_revision'],
    'started_utc': transformer_started,
    'finished_utc': datetime.now(timezone.utc).isoformat(),
}
if globals().get('summary_benchmark_complete', False):
    runtime['summary_benchmark'] = {
        'completed': True,
        'cache_archive_url': summary_cache_url,
        'cache_archive_sha256': summary_cache_sha256,
        'cache_manifest_sha256': hashlib.sha256(manifest_bytes).hexdigest(),
        'encoder_source_sha256': summary_cache_manifest['encoder_source_sha256'],
        'packages': {name: version(name) for name in ('rouge-score', 'nltk', 'rapidfuzz')},
    }
else:
    runtime['summary_benchmark'] = {'completed': False}
(transformer_final / 'runtime.json').write_text(json.dumps(runtime, indent=2) + '\n')

bundle_files = [
    (transformer_final / name, name)
    for name in ('role_labeller.pt', 'protocol.json', 'selection.json',
                 'training_history.json', 'runtime.json')
]
for candidate in selection['candidates']:
    relative = f"candidate-{candidate['candidate']}/history.json"
    bundle_files.append((transformer_tuning / relative, 'validation/' + relative))
for name in ('evaluation.json', 'role_metrics.json', 'test_predictions.json'):
    bundle_files.append((transformer_evaluation / name, 'test-evaluation/' + name))
if globals().get('summary_benchmark_complete', False):
    bundle_files.extend((summary_benchmark / name, 'summary-benchmark/' + name)
                        for name in summary_benchmark_reports)
assert all(path.is_file() for path, _ in bundle_files), 'Complete all prior stages before exporting.'
assert all('validation_labeller.pt' not in name and 'checkpoint.pt' not in name
           for _, name in bundle_files)

result = transformer_work / 'transformer_role_labeller.zip'
with ZipFile(result, 'w', compression=ZIP_DEFLATED) as archive:
    for path, name in bundle_files:
        archive.write(path, name)
digest = hashlib.sha256()
with result.open('rb') as stream:
    for chunk in iter(lambda: stream.read(1024 * 1024), b''):
        digest.update(chunk)
print('Archive:', result)
print('Archive SHA256:', digest.hexdigest())
files.download(str(result))

## Use the downloaded model locally

Extract `transformer_role_labeller.zip` into `runs/transformer-colab` in an updated repository checkout. Keep the bundled held-out report; do not repeat test-based selection. If you ran the optional hosted summary benchmark, read `runs/transformer-colab/summary-benchmark/metrics.csv`. Otherwise, install the local Python 3.11 dependencies using the README and run:

```sh
python -m legal_lexrank --role-run runs/transformer-colab --output runs/transformer-benchmark
```

The benchmark scores the ten test judgments with **fixed default role weights** and an **unweighted** baseline, at 30%, 40% and 50% sentence budgets, for TF-IDF, S-BERT, Legal-BERT and OpenAI embeddings. Uncached OpenAI embeddings require `OPENAI_API_KEY`; the optional hosted cell reuses verified similarities and needs no key. For an uncached local run, use `--backends tfidf sbert legalbert` to run the other three. The role classifier itself needs no API key.

The default role weights stay at 0.10 Facts, 0.10 Argument, 0.10 Statute, 0.15 Precedent, 0.15 Ratio of the decision, 0.20 Ruling by Present Court and 0.20 Ruling by Lower Court.